# FDRI GEAR-Hourly Zarr Data in R
### Using the native `zarr` R package

**Launch this notebook in your preferred environment:**

| Environment | Link |
|---|---|
| [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_ORG/YOUR_REPO/blob/main/fdri_gearhrly_zarr.ipynb) | Google Colab (R runtime) |
| [![Launch in JupyterLite](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://jupyter.org/try-jupyter/lab/?path=fdri_gearhrly_zarr.ipynb) | JupyterLite (browser, no install) |
| [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/YOUR_ORG/YOUR_REPO/HEAD?labpath=fdri_gearhrly_zarr.ipynb) | Binder (free server-based Jupyter) |
| [![Open in DataLabs](https://img.shields.io/badge/Open%20in-NERC%20DataLabs-blue?logo=jupyter)](https://datalabs.nerc.ac.uk) | NERC DataLabs (JASMIN) |

> **Replace `YOUR_ORG/YOUR_REPO`** in the Colab and Binder URLs above with your GitHub repository path once the notebook is published.

---

## About this notebook

This notebook explores **GEAR-Hourly** (Gridded Estimates of Areal Rainfall — Hourly), a gridded rainfall dataset for Great Britain produced as part of the [FDRI (Floods and Droughts Research Infrastructure)](https://fdri.org.uk) project hosted on JASMIN object storage.

We use the **[`zarr`](https://r-cf.github.io/zarr/)** package — a native, extensible R driver for Zarr v3 — to:
1. Open the remote HTTP Zarr store
2. Inspect store structure and metadata
3. Read coordinate arrays (time, lat, lon)
4. Extract and plot a time series at a single grid point
5. Extract and map a spatial slice

**Data URL:**
```
https://fdri-o.s3-ext.jc.rl.ac.uk/gearhrly/gearhrly_15day_100km_chunks.zarr
```

### Runtime compatibility

| Runtime | Status | Notes |
|---|---|---|
| Quarto (local R) | ✅ Full | `knitr` engine; all packages install normally |
| Jupyter + IRkernel (local) | ✅ Full | Standard R install |
| Google Colab (R runtime) | ✅ Full | Switch runtime to R; install cell runs automatically |
| NERC DataLabs / JASMIN | ✅ Full | Preferred for large data access (co-located) |
| Binder | ✅ Full | Free but slow cold-start; add `runtime.txt` + `install.R` |
| JupyterLite (webR) | ⚠️ Partial | `zarr` package may not compile in WASM; use Binder if this fails |

---

## Quarto users

If running this as a `.qmd` file, add the following YAML header:

```yaml
---
title: "FDRI GEAR-Hourly Zarr in R"
format:
  html:
    code-fold: true
    toc: true
execute:
  echo: true
  warning: false
engine: knitr
---
```

---
## 0. Setup

Install and load the `zarr` package (CRAN) plus plotting libraries.
On first run this takes ~1–2 minutes. The `zarr` package requires the `curl` package for HTTP stores.

In [ ]:
# ── Install packages ──────────────────────────────────────────────────────────
pkgs <- c("zarr", "curl", "ggplot2", "dplyr", "tidyr", "sf", "rnaturalearth",
          "rnaturalearthdata", "scales", "lubridate")
new  <- pkgs[!pkgs %in% installed.packages()[, "Package"]]
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org")

# ── Load ──────────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
  library(zarr)             # Native Zarr v3 driver
  library(ggplot2)          # Plotting
  library(dplyr)            # Data wrangling
  library(tidyr)
  library(sf)               # Spatial features
  library(rnaturalearth)    # Country/coastline outlines
  library(rnaturalearthdata)
  library(scales)           # Axis formatting
  library(lubridate)        # Date handling
})

cat("zarr version:", as.character(packageVersion("zarr")), "\n")
cat("✓ All packages loaded\n")

---
## 1. Open the Zarr store

The `zarr` package opens HTTP stores with `open_zarr()`. For Zarr v2 stores that have a consolidated `.zmetadata` file at the root, the package automatically reads the full hierarchy. Otherwise it queries the store's root for a `zarr.json` (v3) or `.zarray` (v2) file.

> **Note**: The `zarr` package handles HTTP stores as **read-only** automatically.

In [ ]:
# ── Store URL ─────────────────────────────────────────────────────────────────
store_url <- "https://fdri-o.s3-ext.jc.rl.ac.uk/gearhrly/gearhrly_15day_100km_chunks.zarr"

# Open the store — zarr handles HTTP natively, no credentials needed for public stores
z <- open_zarr(store_url)

# Print the top-level summary
print(z)

---
## 2. Inspect the store hierarchy and metadata

### 2a. Hierarchy

The `$hierarchy()` method shows all groups and arrays in the store, similar to `ncdump -h` for NetCDF or `xr.open_zarr(url)` in Python.

In [ ]:
# Print the full store hierarchy (groups ☰ and arrays ⌗)
z$hierarchy()

In [ ]:
# Programmatic access: list all array paths
cat("=== Array paths ===\n")
cat(paste(" •", z$arrays), sep = "\n")

cat("\n=== Group paths ===\n")
cat(paste(" •", z$groups), sep = "\n")

### 2b. Root (global) attributes

Global attributes on the root group are the Zarr equivalent of NetCDF global attributes.

In [ ]:
# Get the root group's attributes
root_attrs <- z[["//"]]$attributes

if (length(root_attrs) == 0) {
  cat("No root-level attributes found (check individual array attributes below)\n")
} else {
  cat("=== ROOT ATTRIBUTES ===\n")
  for (nm in names(root_attrs)) {
    val <- paste(unlist(root_attrs[[nm]]), collapse = ", ")
    if (nchar(val) > 100) val <- paste0(substr(val, 1, 97), "...")
    cat(sprintf("  %-30s : %s\n", nm, val))
  }
}

### 2c. Inspect a specific array

Arrays are accessed using list-like `z[["path"]]` syntax. Each array object exposes its shape, chunking, data type, and attributes.

In [ ]:
# ── Inspect each array in the store ──────────────────────────────────────────
# We use z$arrays to iterate; path must start with "/"

for (arr_path in z$arrays) {
  arr <- z[[arr_path]]
  cat(sprintf("\n╔══ %s ══╗\n", arr_path))
  cat(sprintf("  Data type : %s\n", arr$data_type$data_type))
  cat(sprintf("  Shape     : %s\n", paste(arr$shape, collapse = " × ")))
  cat(sprintf("  Chunks    : %s\n", paste(arr$chunk_grid$chunk_shape, collapse = " × ")))
  
  # Print any attributes
  attrs <- arr$attributes
  if (length(attrs) > 0) {
    cat("  Attributes:\n")
    for (nm in names(attrs)) {
      val <- paste(unlist(attrs[[nm]]), collapse = ", ")
      if (nchar(val) > 80) val <- paste0(substr(val, 1, 77), "...")
      cat(sprintf("    %-20s : %s\n", nm, val))
    }
  }
}

---
## 3. Read coordinate arrays

Zarr arrays are indexed from 1 in R (the package handles the 0-based spec internally). We use `arr[NULL, NULL, ...]` (empty index = all elements) to read an entire coordinate array efficiently.

In [ ]:
# ── Read coordinate arrays ────────────────────────────────────────────────────
# Adjust these paths to match what z$arrays printed above
# Common names: /time, /lat, /lon, /latitude, /longitude, /x, /y

read_coord <- function(z, path) {
  tryCatch({
    arr <- z[[path]]
    # Empty index reads all values
    as.numeric(arr[])
  }, error = function(e) {
    message("  Could not read '", path, "': ", conditionMessage(e))
    NULL
  })
}

cat("Reading coordinates...\n")
lons  <- read_coord(z, "/longitude")
lats  <- read_coord(z, "/latitude")
times <- read_coord(z, "/time")

# ── Decode time axis ──────────────────────────────────────────────────────────
# Get units from the time array's attributes
time_arr   <- z[["/time"]]
time_attrs <- time_arr$attributes
time_units <- time_attrs[["units"]]
time_cal   <- time_attrs[["calendar"]]

cat("Time units  :", time_units, "\n")
cat("Calendar    :", if (is.null(time_cal)) "(not specified, assuming gregorian)" else time_cal, "\n")

# Parse "hours since YYYY-MM-DD HH:MM:SS" style CF units
origin_str <- sub(".*since\\s+", "", time_units)
origin_dt  <- as.POSIXct(origin_str, tz = "UTC",
                format = ifelse(grepl(":", origin_str), "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"))

# Determine units multiplier (hours → seconds; days → seconds; etc.)
unit_word  <- tolower(sub("\\s+since.*", "", time_units))
multiplier <- switch(unit_word,
  "hours"   = 3600,
  "days"    = 86400,
  "minutes" = 60,
  "seconds" = 1,
  3600  # default to hours
)

datetimes <- origin_dt + as.numeric(times) * multiplier

cat(sprintf("\nLongitude   : %d values, %.3f° to %.3f°\n",
  length(lons), min(lons), max(lons)))
cat(sprintf("Latitude    : %d values, %.3f° to %.3f°\n",
  length(lats), min(lats), max(lats)))
cat(sprintf("Time steps  : %d, from %s to %s\n",
  length(datetimes), format(min(datetimes)), format(max(datetimes))))

---
## 4. Choose the data variable

Set `data_var` to the name of the rainfall (or other) variable in the store.

In [ ]:
# Set to the main data variable — adjust based on what z$arrays showed above
# Likely candidates: "rainfall_amount", "precip", "rain", "rr"
data_var <- "rainfall_amount"   # <-- adjust if needed

data_arr   <- z[[paste0("/", data_var)]]
data_attrs <- data_arr$attributes

cat("=== Data variable:", data_var, "===\n")
cat("Shape  :", paste(data_arr$shape, collapse = " × "), "\n")
cat("Chunks :", paste(data_arr$chunk_grid$chunk_shape, collapse = " × "), "\n")

# Print attributes
for (nm in names(data_attrs)) {
  cat(sprintf("  %-20s : %s\n", nm, paste(unlist(data_attrs[[nm]]), collapse = ", ")))
}

# Extract units and long name for plot labels
var_units <- data_attrs[["units"]] %||% ""
var_long  <- data_attrs[["long_name"]] %||% data_attrs[["standard_name"]] %||% data_var

---
## 5. Time series at a single grid point

We use R's standard array indexing on the Zarr array object. The `zarr` package translates this to efficient HTTP range requests — only the relevant chunks are fetched.

### Chunking note
This store uses 15-day × 100km chunks (as indicated in the filename). A single point time series will download only the time chunks that contain our requested steps.

In [ ]:
# ── Choose a target location ──────────────────────────────────────────────────
# Default: Edinburgh, Scotland
target_lon <- -3.19
target_lat <- 55.95

# Find the nearest grid cell indices (1-based for R)
i_lon <- which.min(abs(lons - target_lon))
i_lat <- which.min(abs(lats - target_lat))

cat(sprintf("Target location  : lon=%.3f, lat=%.3f\n", target_lon, target_lat))
cat(sprintf("Nearest grid cell: lon=%.3f (index %d), lat=%.3f (index %d)\n",
  lons[i_lon], i_lon, lats[i_lat], i_lat))

In [ ]:
# ── Read the full time series for this grid cell ──────────────────────────────
# Zarr array indexing in R: arr[dim1_indices, dim2_indices, dim3_indices, ...]
# Empty index (or NULL) = all elements along that dimension
# Dimension order: check data_arr$shape against coord lengths to confirm

n_time <- length(times)

cat(sprintf("Reading %d time steps for grid cell [%d, %d]...\n",
  n_time, i_lat, i_lon))

# Typical xarray-written Zarr dim order: [time, latitude, longitude]
# Adjust index order if your array has a different convention
ts_raw <- data_arr[1:n_time, i_lat, i_lon]
ts_vals <- as.numeric(ts_raw)

# Fill value / NA handling (zarr package auto-converts fill_value to NA)
cat(sprintf("NA count: %d / %d (%.1f%%)\n",
  sum(is.na(ts_vals)), n_time, 100 * mean(is.na(ts_vals))))

# Build a data frame
ts_df <- data.frame(
  datetime = datetimes,
  value    = ts_vals
)
head(ts_df)

In [ ]:
# ── Plot: full time series ────────────────────────────────────────────────────
y_label <- if (nchar(var_units) > 0) paste0(var_long, "\n(", var_units, ")") else var_long
subtitle_ts <- sprintf("Grid cell: lon=%.3f°, lat=%.3f°", lons[i_lon], lats[i_lat])

p_ts <- ggplot(ts_df, aes(x = datetime, y = value)) +
  geom_line(colour = "#1d6fa4", linewidth = 0.5, alpha = 0.85) +
  scale_x_datetime(labels = scales::date_format("%b %Y")) +
  scale_y_continuous(labels = scales::comma) +
  labs(
    title    = paste("FDRI GEAR-Hourly:", var_long),
    subtitle = subtitle_ts,
    x = NULL,
    y = y_label,
    caption  = "Source: FDRI / UKCEH · fdri-o.s3-ext.jc.rl.ac.uk"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title    = element_text(face = "bold", size = 13),
    plot.subtitle = element_text(colour = "grey40"),
    panel.grid.minor = element_blank(),
    plot.caption  = element_text(colour = "grey60", size = 8)
  )

print(p_ts)

In [ ]:
# ── Plot: daily aggregation (sum per day for rainfall) ────────────────────────
ts_daily <- ts_df |>
  mutate(date = as.Date(datetime)) |>
  group_by(date) |>
  summarise(daily_total = sum(value, na.rm = TRUE), .groups = "drop")

p_daily <- ggplot(ts_daily, aes(x = date, y = daily_total)) +
  geom_col(fill = "#1d6fa4", width = 1, alpha = 0.8) +
  scale_x_date(labels = scales::date_format("%b %Y")) +
  labs(
    title    = paste("Daily total:", var_long),
    subtitle = subtitle_ts,
    x = NULL,
    y = paste0("Daily total (", var_units, ")"),
    caption  = "Source: FDRI / UKCEH"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title    = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey40"),
    panel.grid.minor = element_blank(),
    plot.caption  = element_text(colour = "grey60", size = 8)
  )

print(p_daily)

---
## 6. Spatial map at a single time step

Read the full spatial grid for one time step. With Zarr's chunk-aware access, this fetches only the chunks needed for a 2-D slice — much more efficient than downloading the entire array.

In [ ]:
# ── Choose a time step ────────────────────────────────────────────────────────
t_index    <- 1   # first time step; change as needed
slice_time <- format(datetimes[t_index], "%Y-%m-%d %H:%M UTC")
cat(sprintf("Reading spatial slice for time step %d: %s\n", t_index, slice_time))

n_lat <- length(lats)
n_lon <- length(lons)

# Read the 2-D slice: [single_time, all_lat, all_lon]
slice_raw <- data_arr[t_index, 1:n_lat, 1:n_lon]
slice_mat <- matrix(as.numeric(slice_raw), nrow = n_lat, ncol = n_lon)

cat(sprintf("Grid: %d lats × %d lons\n", n_lat, n_lon))
cat(sprintf("Value range: %.3f to %.3f %s\n",
  min(slice_mat, na.rm = TRUE), max(slice_mat, na.rm = TRUE), var_units))

In [ ]:
# ── Build long-format data frame for ggplot2 ──────────────────────────────────
map_df <- expand.grid(lon = lons, lat = lats)
map_df$value <- as.vector(t(slice_mat))  # transpose: expand.grid is lon-major

# Get GB/Ireland coastline for overlay
gb <- ne_countries(country = c("United Kingdom", "Ireland"),
                   scale = "medium", returnclass = "sf")

# Clip to the store's geographic extent
lon_range <- range(lons)
lat_range <- range(lats)

p_map <- ggplot() +
  geom_raster(data = map_df,
              aes(x = lon, y = lat, fill = value), na.rm = TRUE) +
  geom_sf(data = gb, fill = NA, colour = "grey20", linewidth = 0.3) +
  scale_fill_distiller(
    palette  = "YlGnBu",
    direction = 1,
    name     = var_units,
    na.value = "grey90",
    trans    = "sqrt",   # sqrt transform helps visualise rainfall (right-skewed)
    labels   = scales::comma
  ) +
  coord_sf(
    xlim = lon_range,
    ylim = lat_range,
    expand = FALSE
  ) +
  labs(
    title    = paste("FDRI GEAR-Hourly:", var_long),
    subtitle = slice_time,
    x = "Longitude", y = "Latitude",
    caption  = "Source: FDRI / UKCEH · fdri-o.s3-ext.jc.rl.ac.uk"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title    = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey40"),
    legend.position = "right",
    panel.grid    = element_line(colour = "grey80", linewidth = 0.2),
    plot.caption  = element_text(colour = "grey60", size = 8)
  )

print(p_map)

---
## 7. Bonus: Mean over a time window

Compute the spatial mean over the first N time steps. We read only those time chunks — this is where Zarr's chunked access pattern really shines compared to NetCDF.

In [ ]:
# ── Read first 24 time steps (e.g. one day of hourly data) ───────────────────
n_steps <- min(24, n_time)
cat(sprintf("Reading %d time steps (%s to %s)...\n",
  n_steps, format(datetimes[1]), format(datetimes[n_steps])))

# Read [time_window, all_lat, all_lon]
window_raw  <- data_arr[1:n_steps, 1:n_lat, 1:n_lon]

# Reshape to [n_steps, n_lat * n_lon] then compute column-wise mean
window_arr  <- array(as.numeric(window_raw), dim = c(n_steps, n_lat, n_lon))
mean_grid   <- apply(window_arr, c(2, 3), mean, na.rm = TRUE)
total_grid  <- apply(window_arr, c(2, 3), sum,  na.rm = TRUE)

mean_df        <- expand.grid(lon = lons, lat = lats)
mean_df$mean   <- as.vector(t(mean_grid))
mean_df$total  <- as.vector(t(total_grid))

p_mean <- ggplot() +
  geom_raster(data = mean_df,
              aes(x = lon, y = lat, fill = total), na.rm = TRUE) +
  geom_sf(data = gb, fill = NA, colour = "grey20", linewidth = 0.3) +
  scale_fill_distiller(
    palette   = "YlGnBu",
    direction = 1,
    name      = paste0("Total\n(", var_units, ")"),
    na.value  = "grey90",
    trans     = "sqrt",
    labels    = scales::comma
  ) +
  coord_sf(xlim = lon_range, ylim = lat_range, expand = FALSE) +
  labs(
    title    = paste0("FDRI GEAR-Hourly: ", n_steps, "-hour accumulated ", var_long),
    subtitle = sprintf("%s to %s",
      format(datetimes[1], "%Y-%m-%d %H:%M UTC"),
      format(datetimes[n_steps], "%Y-%m-%d %H:%M UTC")),
    x = "Longitude", y = "Latitude",
    caption  = "Source: FDRI / UKCEH · fdri-o.s3-ext.jc.rl.ac.uk"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title    = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey40"),
    legend.position = "right",
    panel.grid    = element_line(colour = "grey80", linewidth = 0.2),
    plot.caption  = element_text(colour = "grey60", size = 8)
  )

print(p_mean)

---
## 8. Session info

Useful for reproducibility — records the exact package versions used.

In [ ]:
sessionInfo()

---
## Troubleshooting & tips

| Symptom | Fix |
|---------|-----|
| `open_zarr()` hangs or times out | Check you have internet access; try `curl::curl_fetch_memory(store_url)` to test connectivity |
| Store opens but `z$arrays` is empty | The store may use consolidated metadata differently — try `z$hierarchy()` to see what was found |
| Variable name not found | Run section 2 to see actual array paths; update `data_var` |
| Dimension order is wrong (map looks transposed) | Swap indices in `data_arr[t, lat, lon]` to `data_arr[t, lon, lat]` |
| Time decoding gives wrong dates | Check `time_units` output and adjust `multiplier` manually |
| Large slice is slow | This is expected over HTTP; either co-locate on JASMIN or narrow the time window |
| `rnaturalearth` error | Install `rnaturalearthdata` with `install.packages("rnaturalearthdata")` |
| JupyterLite / webR: `zarr` fails to compile | Use Binder, Colab, or a local environment instead |

### Quarto rendering

To render as a standalone HTML report:
```bash
quarto render fdri_gearhrly_zarr.ipynb --to html
```
Or convert to `.qmd` first:
```bash
quarto convert fdri_gearhrly_zarr.ipynb
```